##L06: Joins, aggregations and window functions

###Step 1. Check the join before trusting it
The zone lookup must have one row per LocationID, otherwise the join duplicates trips.

In [0]:
%sql
SELECT LocationID, COUNT(*) FROM da.bronze.taxi_zones GROUP BY LocationID HAVING COUNT(*) > 1;
SELECT COUNT(*) AS trips FROM da.silver.yellow_trips_conformed;
SELECT COUNT(*) AS joined FROM da.silver.yellow_trips_conformed t
LEFT JOIN da.bronze.taxi_zones z ON t.pu_location_id = z.LocationID;

###Step 2. Busiest pickup zones

In [0]:
%sql
SELECT 
    z.Borough, 
    z.Zone, 
    COUNT(*) AS trips, 
    ROUND(AVG(t.fare_amount), 2) AS avg_fare
FROM da.silver.yellow_trips_conformed AS t
JOIN da.bronze.taxi_zones z ON t.pu_location_id = z.LocationID
GROUP BY z.Borough, z.Zone 
ORDER BY trips DESC LIMIT 10
;

###Step 3. Top 3 zones per borough (window)

In [0]:
%sql
WITH z AS 
(
  SELECT 
    zn.Borough, 
    zn.Zone, 
    COUNT(*) AS trips
  FROM da.silver.yellow_trips_conformed AS t
  JOIN da.bronze.taxi_zones AS zn ON t.pu_location_id = zn.LocationID
  GROUP BY zn.Borough, zn.Zone
)

SELECT * FROM 
(
  SELECT *, ROW_NUMBER() OVER (PARTITION BY Borough ORDER BY trips DESC) AS rn FROM z
)
WHERE rn <= 3 
ORDER BY Borough, rn
;

###Step 4. Day-over-day change and running total

In [0]:
%sql
WITH d AS 
(
    SELECT 
        pickup_date, 
        COUNT(*) AS trips 
    FROM da.silver.yellow_trips_conformed
    WHERE pickup_date BETWEEN '2024-12-01' AND '2025-02-28' 
    GROUP BY pickup_date
)

SELECT 
    pickup_date, 
    trips,
    trips - LAG(trips) OVER (ORDER BY pickup_date) AS change_vs_prev_day,
    SUM(trips) OVER (ORDER BY pickup_date) AS running_total
    
FROM d ORDER BY pickup_date;

###Step 5. Find orphans with an anti join
Trips whose pickup zone is not in the lookup.

In [0]:
%sql
SELECT 
    COUNT(*) AS orphan_trips 
FROM da.silver.yellow_trips_conformed AS t
LEFT ANTI JOIN da.bronze.taxi_zones z ON t.pu_location_id = z.LocationID
;

###Step 6. Same top-zones query in PySpark
Rewrite step 2 in PySpark (join, groupBy, orderBy) and confirm the same top zone. Compare the two with explain().

In [0]:
from pyspark.sql import functions as F

trips = spark.table("da.silver.yellow_trips_conformed")
zones = spark.table("da.bronze.taxi_zones")

# Step 2 rewritten in PySpark: join, groupBy, orderBy, limit
top_zones_py = (trips
    .join(zones, trips.pu_location_id == zones.LocationID)
    .groupBy("Borough", "Zone")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy(F.desc("trips"))
    .limit(10))

# The exact same query in SQL, for a side-by-side check
top_zones_sql = spark.sql("""
    SELECT z.Borough, z.Zone, COUNT(*) AS trips, ROUND(AVG(t.fare_amount), 2) AS avg_fare
    FROM da.silver.yellow_trips_conformed AS t
    JOIN da.bronze.taxi_zones z ON t.pu_location_id = z.LocationID
    GROUP BY z.Borough, z.Zone
    ORDER BY trips DESC
    LIMIT 10
""")

print("Top zone (PySpark):", top_zones_py.first().asDict())
print("Same top zone as SQL:", top_zones_py.first() == top_zones_sql.first())

display(top_zones_py)

# Both compile to the same physical plan — the DataFrame API and SQL are
# two syntaxes over one engine; only the front-end differs.
print("---- PySpark DataFrame plan ----")
top_zones_py.explain()
print("---- SQL plan ----")
top_zones_sql.explain()